# 🎲 Homework 7: Probability, Bootstrapping, Bias-Variance Tradeoff, Logistic Regression
## Due Date: Friday, November 20th, 11:59 PM PDT

‼️ Note: There is also a written portion of this HW! Be sure to complete both this notebook and the written assignment. **This written assignment is different from the Coding Written assignment submitted automatically with the Coding assignment.**

You must submit this assignment to Pensive by the on-time deadline, Friday, November 20th, 11:59 PM PDT. Please read the syllabus for the Slip Day policy. No late submissions beyond what is outlined in the Slip Day policy will be accepted. **We strongly encourage you to submit your work to Pensive several hours before the stated deadline.** This way, you will have ample time to reach out to staff for support if you encounter difficulties with submission. While course staff is happy to help guide you with submitting your assignment ahead of the deadline, we will not respond to last-minute requests for assistance.

Please read the instructions carefully when submitting your work to Pensive. 

This is one part of a two-part assignment. After completing this part ("Homework 7 Coding"), also remember to complete the "Homework 7 Handwritten Work" assignment, which will be about random variables and logistic regression.


## Collaboration Policy

Data science is a collaborative activity. While you may talk with others about the homework, we ask that you **write your solutions individually**. If you do discuss the assignments with others please **include their names** below.

**Collaborators**: *list collaborators here*

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw07.ipynb")

## Introduction

While working on this notebook, you will gain experience with the following:
* Bootstrap sampling,
* The bias-variance tradeoff and decomposition,
* Multicollinearity in features, and
* Logistic Regression

## Grading
Grading is broken down into auto-graded answers and free responses. 

For auto-graded answers, the results of your code are compared to provided tests.

For free response, readers will evaluate how well you answered the question and/or fulfilled the requirements of the question. 

### Score breakdown

Question | Manual | Points
--- |---| ---
1a| Yes | 2
1b| Yes | 2
1c| Yes | 2
1d| Yes | 2
1e| Yes | 1
2a| No | 3
2b| Yes |3
2c| Yes | 1
3a| No | 2
3b| Yes | 2
3c| Yes | 2
3d(i)| Yes | 2
3d(ii)| Yes | 2
4a| Yes | 1
4b| Yes | 2
5a | No | 1
5bi | No | 2 |
5bii | Yes | 1
5c | No | 2
5d | No | 2
5e | No | 2
5f | Yes | 3
5g | Yes | 3
**Total** |  | **45**

If you run into any technical issues, we highly recommend checking out the [Data 100 Debugging Guide](https://ds100.org/debugging-guide/).

In [ ]:
# Run this cell to set up your notebook
import numpy as np
import polars as pl
import sklearn.linear_model as lm
from sklearn.model_selection import train_test_split
import sklearn
import sklearn.datasets

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown

import scipy.stats
from hw07_utils import visualize_gradient_descent, plot_function, generate_X, generate_Y

import warnings
warnings.filterwarnings("ignore")

pl.Config.set_float_precision(3)

<br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 📈 Question 1: Exploratory Data Analysis

In questions 1 through 3, we will explore a dataset of the 2022 U.S. News & World Report rankings for liberal arts colleges.
The dataset in this HW is a simplified version of the [original data source](https://andyreiter.com/datasets/). (Note: We are using the 2022 version, not the updated 2026 version.)

Let's perform some initial EDA to identify trends and patterns in this dataset.

In [ ]:
# Run this cell to load the data; no further action is needed.
college_data = pl.read_csv('data/2022_college_rankings.csv')
college_data.head(5)

The granularity of the dataset is an individual **college**; each row includes metrics, ratings, and features of **one college in the U.S**.

Here are the columns in the dataset:
* `Institution`: The name of the college.
* `Type`: Whether the college is `'private'` or `'public'`.
* `Overall Score (0-100)`: The score computed by U.S. News & World Report to generate the ranking. **This is our outcome variable.**
* The remaining columns contain features that are related to each college's overall score. 

We can use `college_data.glimpse()` and `college_data.describe()` to see various statistics about the features of the provided college ranking data. Do any particular statistics stand out to you? Which might be useful when modeling?

**Note:** This isn't a question, so it's not worth any points. This is just food for thought as you start to explore the dataset.

In [ ]:
college_data.glimpse()

In [ ]:
college_data.describe()

You may have observed that `describe()` doesn't report a mean or standard deviation for several features, only counts and min/max. This is because those features are non-numeric. For example, some features are in the string format `"XX%"`.

Before we can fit a linear model accounting for all features in the dataset, we have to convert all features to a numeric representation.

<!-- BEGIN QUESTION -->

<br>

---

### Question 1a

Below, we provide code that generates `cleaned_college_data`. Write a comment explaining the purpose of the block that follows.

In [ ]:
cleaned_college_data = college_data

# Write your explanation here.
for col in college_data.columns:
  if cleaned_college_data[col].is_null().any():
    print(col)
    cleaned_college_data = cleaned_college_data.with_columns(
        pl.col(col).is_null().alias(col + '_missing')
    )


<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br>

---

### Question 1b

Now, we will convert all non-numeric columns of `cleaned_college_data` into numbers. Additionally, we will create some new columns to encode additional information.

Below, we provide code that transforms `cleaned_college_data`. At each numbered location, write a comment explaining the purpose of the line or block that follows.

In [ ]:
# 1. Write your explanation here.
string_cols = cleaned_college_data.select(pl.col(pl.String))

# 2. Write your explanation here.
cleaned_college_data = cleaned_college_data.with_columns(
    (pl.col("Type") == "public").cast(pl.Int64).alias("Is Public"),
    (pl.col("Type") == "private").cast(pl.Int64).alias("Is Private")
)

# 3. Write your explanation here.
for col in string_cols.columns:
  if cleaned_college_data[col].str.contains('%').any():
    cleaned_college_data = cleaned_college_data.with_columns(
        pl.col(col).str.replace_all('%', '').cast(pl.Float64)
    )

# 4. Write your explanation here.
cleaned_college_data = cleaned_college_data.drop(["Institution", "Type"])

# 5. Write your explanation here.
for col in cleaned_college_data.columns:
  if cleaned_college_data[col].is_null().any():
    m = cleaned_college_data[col].mean()
    cleaned_college_data = cleaned_college_data.with_columns(pl.col(col).fill_null(m))


cleaned_college_data

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br>

---

### Question 1c

Linear models are sensitive to multicollinearity among the columns of the design matrix. So, let's determine the extent of multicollinearity in the college rankings dataset.

In the following cell, we provide `arranged_cleaned_college_data`, a copy of `cleaned_college_data` containing ten selected columns arranged in a particular order.

Using `corr_matrix`, create a heatmap displaying the pairwise correlations among all ten variables in `arranged_cleaned_college_data`. **You may use an LLM to help you perform this task.**

Your final plot should:

- Display a $10 \times 10$ correlation heatmap.
- Use `annot=True` so that the correlation values appear in the heatmap.
- Keep the row and column labels on the heatmap.
- Include a descriptive title.

In [ ]:
# Run this cell to create the DataFrame used in Question 1c.
arranged_cleaned_college_data = cleaned_college_data.select(
    [
        "Overall Score (0-100)",
        "Peer Assessment Score (1-5)",
        "Predicted 6yr graduation rate",
        "Actual 6yr graduation rate",
        "Graduation and retention rank",
        "Student Excellence rank",
        "Acceptance rate",
        "Financial resources rank",
        "Is Public",
        "Is Private"
    ]
)

In [ ]:
corr_matrix = arranged_cleaned_college_data.corr()
...


<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br>

---

### Question 1d

Do you notice any patterns in the plot from part (c)? What might explain these patterns? Comment and hypothesize on at least two patterns you notice.

Here are some example questions to ponder:

1. Why do some feature-pairs have correlations of $\pm 1$? Is this a problem?
2. What does the correlation between pairs of features (i.e., graduation-related statistics) look like? Is the magnitude of any of the correlations problematically close to 1?
3. Are any features particularly strong predictors of the outcome? If so, why do you think this is the case?
4. Do any features seem potentially redundant? In other words, do you suspect that any features provide similar information about the outcome as other features?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br>

---
### Question 1e
If we tried to fit a linear regression model with an intercept term using all features in `cleaned_college_data`, we might run into some problems when fitting our model.
The Data 100 staff suggests that we perform the following operation to the `DataFrame` before fitting a model:

In [ ]:
# You must run this cell to pass the public tests for later questions.
cleaned_college_data = cleaned_college_data.drop('Is Private')

Describe the reasoning behind this operation. What problem(s) do we avoid by removing the `Is Private` column from the model fitting process?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 🏗️ Question 2: Constructing a Preliminary College Ranking Model

In this question, you will fit a linear regression model to predict the U.S. News liberal arts college scores based on college metrics. Later, we will investigate the bias, variance, and observational noise of this model.

<br>

---

### Question 2a

Train an `sklearn` OLS model that predicts each college's overall score using all of the other columns in `cleaned_college_data`. Include an intercept term in the model. **As a reminder, you should not use LLMs on this assignment except for questions where we specifically suggest using them.**

Use `train_test_split()` [(documentation)](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) to evaluate your model's RMSE on a held-out test set containing 33% of the college data; call the resulting splits `X_train`, `X_test`, `Y_train`, and `Y_test`.

You can instantiate your linear regression model using `lm.LinearRegression` [(documentation)](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

**Hint**: You may want to use `DataFrame.drop` in your answer. For example, `df.drop("column_name")` removes the column named `"column_name"` from `df`. For more information, check out the [Documentation for `DataFrame.drop`.](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.drop.html)

In [ ]:

# Create train/test sets
X = cleaned_college_data.drop(...)
Y = cleaned_college_data[...]

X_train, X_test, Y_train, Y_test = train_test_split(
    ...,
    ...,
    test_size=...,
    random_state=100
)

# Fit the linear model
model = lm.LinearRegression()
model.fit(..., ...)

# Compute RMSE on the train and test sets
train_rmse_cpc = np.sqrt(
    (...).mean()
)
test_rmse_cpc = np.sqrt(
    (...).mean()
)

# Ignore these two lines of code; they are used in a later question.
X_original = X
Y_original = Y

train_rmse_cpc, test_rmse_cpc

In [ ]:
grader.check("q2a")

<!-- BEGIN QUESTION -->

<br>

---

### Question 2b

Let's visualize the model performance from part 2(a). Plot the following:
1. The observed values vs. the predicted values on the test set.
2. The residuals plot. **Recall that for multiple linear regression, we plot the residuals against the predicted values.**
For both plots, place the predicted college scores on the horizontal axis.

In the first plot:

- Plot the predicted college scores against the observed college scores.
- The provided diagonal red line represents $\hat{Y}=Y$, where the prediction equals the observed value.
- Fill in the axis labels and title.

In the second plot:

- Plot the residuals against the predicted college scores.
- Calculate each residual as the observed value minus the predicted value, $Y-\hat{Y}$.
- Complete the provided horizontal reference line so that it represents a residual of 0.
- Fill in the axis labels and title.

We have provided the figure size, subplot positions, reference-line code, x-axis limits, and `plt.tight_layout()`. Do not modify the lines marked `Do not change this line`.

**Note:** To make the plots easier to compare, both plots use the same x-axis limits.

You may find the following documentation helpful:

- [`sns.scatterplot`](https://seaborn.pydata.org/generated/seaborn.scatterplot.html)
- [`plt.plot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html)
- [`plt.axhline`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.axhline.html)

In [ ]:

plt.figure(figsize=(12, 6))      # Do not change this line.

# Plot observed values against predicted values.
plt.subplot(121)                 # Do not change this line.

Y_pred = model.predict(...)

sns.scatterplot(...)
# Plot the reference line y = x.
plt.axline((0, 0), slope=1, color='red', linestyle='--')
plt.xlim(40, 100)
plt.ylim(40, 100)
plt.ylabel(...)
plt.xlabel(...)
plt.title(...)

# Plot residuals against predicted values.
plt.subplot(122)                 # Do not change this line.

sns.scatterplot(...)
plt.axhline(y=..., color="r")
plt.xlim(40, 100)
plt.ylabel(...)
plt.xlabel(...)
plt.title(...)

plt.tight_layout()               # Do not change this line.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br>

---

### Question 2c

Describe what the plots in part (b) indicate about this linear model. In particular, are the predictions good, and do the residuals appear uncorrelated with the predictions?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 🤹 Question 3: Confidence Intervals and Regression Coefficients

We'll next analyze the statistical significance of coefficients in our linear model. Specifically, we will use bootstrapping to get $95\%$ confidence intervals on the fitted parameters. Each step of bootstrapping corresponds to one part of this question:

1. Question 3(a): Assume the sample is representative of the population from which it was drawn. To generate a new, synthetic sample, resample with replacement from the original sample. Generate one bootstrap resample.
2. Question 3(b): Repeat this process `B` times and calculate a statistic for each of the `B` resamples. Possible statistics are the mean, correlation, coefficients of a linear model (Question 3 below), or any function of the sample data.
3. Question 3(c): Construct a confidence interval by grabbing relevant percentiles of the `B` synthetic summary statistics. For example, for an 80% confidence interval, we grab the 10th and 90th percentiles.
4. Question 3(d): Use the resulting confidence interval to make inferences about the population-level statistic.

<br>

---

### Question 3a

Fill in the blank below to implement the `bootstrap_sample()` function. (Our solution requires only one line of code!)

Specification:
- `bootstrap_sample()` returns one bootstrap `DataFrame` generated from the dataset `data`.
- To generate a bootstrapped dataset from an actual dataset with $n$ rows, sample $n$ rows uniformly at random *with replacement* so that the bootstrapped `DataFrame` is the same size as the original `DataFrame`.

**Hint**: Draw the row positions with `np.random.randint()` ([documentation](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randint.html)), then use those positions to select rows from the `DataFrame` ([documentation](https://docs.pola.rs/api/python/stable/reference/dataframe/index.html)).

In [ ]:
def bootstrap_sample(data):
    """
    Performs bootstrap sampling on data to obtain one sample of size n.

    Arguments:
        data - Dataset contained as a DataFrame

    Returns:
        A DataFrame containing one bootstrap sample of size n
    """
    ...

# Print out one bootstrap DataFrame
bootstrap_sample(cleaned_college_data)

In [ ]:
grader.check("q3a")

<!-- BEGIN QUESTION -->

<br>

---

### Question 3b

The `generate_models` function has been provided for you. At each numbered location, write a comment explaining the purpose of the line or block that follows.

**Note:** This question may take a few seconds to run due to the number of models being fit.

In [ ]:
np.random.seed(100) # DO NOT REMOVE THIS LINE

def generate_models(original_df):
    # 1. Write your explanation here.
    datasets = [bootstrap_sample(original_df) for _ in range(1000)]
    models = []
    # 2. Write your explanation here.
    for df in datasets:
        X_b = df.drop("Overall Score (0-100)")
        Y_b = df["Overall Score (0-100)"]
        # 3. Write your explanation here.
        models.append(lm.LinearRegression(fit_intercept = True))
        models[-1].fit(X_b, Y_b)
    # Datasets take up a lot of memory, so we should remove them!
    del datasets
    return models


full_feature_models = generate_models(cleaned_college_data)

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br>

---

### Question 3c

The `confidence_interval` function has been provided for you. At each numbered location, write a comment explaining how the code constructs a 95% confidence interval for each model parameter.
- For a refresher on confidence intervals, refer to [this section in the Data 8 textbook](https://inferentialthinking.com/chapters/13/3/confidence-intervals/).
- Pay close attention to how the arrays used below are formatted. What does each row represent? What does each column represent? To get column `i` from a 2D array, you can use `array[:, i]`.

In [ ]:
def extract_coefs(models, include_intercept = True):
    """
    NOTE: This function has already been implemented. You do not need to modify this!
    
    Extracts coefficients of all the linear regression models in MODELS and returns
    it as a NumPy array with each model's coefficients as one row.
    
    Arguments:
        models - Contains k sklearn LinearRegression models, each with p + 1 coefficients.
        include_intercept - Whether to include an intercept in returned coefficients.
    
    Returns:
        coef_array - Coefficients of all k models, each with p + 1 coefficients (if intercept
                     enabled, otherwise p). The returned object is k x (p + 1) NumPy array.
    """
    coef_array = np.zeros(shape = (len(models), len(models[0].coef_) + 1))
    for i, m in enumerate(models):
        coef_array[i, 0] = m.intercept_
        coef_array[i, 1:] = m.coef_
    if include_intercept:
        return coef_array 
    return coef_array[:, 1:]

def confidence_interval(coefs):
    """
    Calculates confidence intervals for each theta_i based on coefficients of 
    bootstrapped models. Returns output as a list of confidence intervals.
    
    Arguments:
        coefs - Output of extract_coefs, a k x (p + 1) or k x p NumPy array containing
                coefficients of bootstrapped models.
    
    Returns:
        cis - Confidence intervals of each parameter theta_i in the form of a 
              list like this: [(0.5, 0.75), (0.2, 0.4), ...].
    """
    cis = []
    # 1. Write your explanation here.
    for i in range(coefs.shape[1]):
        # 2. Write your explanation here.
        theta_i_values = coefs[:, i]
        # 3. Write your explanation here.
        theta_i_lower_ci, theta_i_upper_ci = np.percentile(theta_i_values, 2.5), np.percentile(theta_i_values, 97.5)
        # 4. Write your explanation here.
        cis.append((theta_i_lower_ci, theta_i_upper_ci))
    return cis


# Compute confidence intervals
np.random.seed(100) # DO NOT REMOVE THIS LINE
full_feature_models_coefs = extract_coefs(full_feature_models)
full_feature_cis = confidence_interval(full_feature_models_coefs)

<!-- END QUESTION -->

<br>

---

### Question 3d
Using the functions provided above, we'll now compute the confidence intervals for the coefficients of two different linear models: Model A and Model B. 

- Recall that Model A uses all features, as defined in `full_feature_models`.

- Model B uses a subset of features. The code below will help you fit Model B's coefficients and compute confidence intervals for them. Instances of Model B will be contained in `partial_feature_models`:

The code below applies the provided `generate_models` function to the features used by Model B. Run the sanity check to confirm that the fitted models use the intended features.

In [ ]:
model_b_features = [
     "Peer Assessment Score (1-5)",
     "Acceptance rate"
]
partial_feature_models = generate_models(
    cleaned_college_data[["Overall Score (0-100)"] + model_b_features]
)
partial_feature_models_coefs = extract_coefs(partial_feature_models)
partial_feature_cis = confidence_interval(partial_feature_models_coefs)

In [ ]:
# For a sanity check, the following lines of code should all output True
print("Coefficient shape is correct:", partial_feature_models_coefs.shape == (1000, 3))
print("Model input feature names are correct:", (partial_feature_models[0].feature_names_in_ == model_b_features).all())

Here's a function that allows us to inspect the confidence interval of each model:

In [ ]:
def print_confidence_intervals(models, cis):
    display(Markdown('##### Confidence Intervals:'))
    md_list = ["|parameter|feature name|lower|upper|",
               "----|----|----|----|"]
    coef_names = np.append(['Intercept'], models[0].feature_names_in_)
    md_list += [r"|$\theta_{" + str(i) + "}" + fr"$|{f_name}|{np.round(lci, 3)}|{np.round(uci, 3)}|" for i, ((lci, uci), f_name) in enumerate(zip(cis, coef_names))]
    display(Markdown('\n'.join(md_list)))

<!-- BEGIN QUESTION -->

#### Question 3d, Part i
Let us first interpret **Model B**, the linear regression model that uses a subset of features from our dataset.

In [ ]:
display(Markdown('#### Model B: Subset of Features'))
print_confidence_intervals(partial_feature_models, partial_feature_cis)

Are $\theta_1$ and $\theta_2$ significantly different from 0 at a 5% significance level? How do you know? 

Does your answer imply that the relationships among `Overall Score (0-100)`, `Peer Assessment Score (1-5)`, and `Acceptance rate` are causal? Do you think the relationships are causal? Explain.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

#### Question 3d, Part ii

In what situation(s) would you prefer a more compact model with just key features, like Model B? On the other hand, in what situation(s) would you want to consider many features, like in Model A? Explain your answer to both of these questions.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ⚖️ Question 4: Performing Bias-Variance Analysis


Recall that the **model variance** on a particular data point ($\vec{x_k}$) is the variance of the model predictions for $\vec{x_k}$ across parallel universes of randomly sampled training datasets:

$$\text{model variance} = \mathrm{Var}(\hat{Y}(\vec{x_k})) = \mathrm{Var}(\vec{x_k}^T\hat{\theta})$$

> Note that $\hat{Y}(\vec{x_k})$ is an alternative way to write $\hat{f}(\vec{x_k})$ from lecture.

To estimate the model variance, we can sample a particular data point $(\vec{x_k}, y_k)$ and calculate the variance of the predictions for $(\vec{x_k}, y_k)$ across parallel sampling universes, where each universe has a model fit to a randomly sampled training dataset. 

- As it turns out, you already have these fitted models stored in `models`!

-  Note that the variance is computed across the different models, which have the same shared structure (i.e., the same input features and transformations). However, fitted parameter values will differ across models since the training data differs across models.

Recall that **model risk** for a single point is the same as the mean squared error (MSE) of $\vec{x_k}$ over our parallel universes of fitted models:

$$
\text{model risk} = \mathbb{E}\left[\left(y_k - \hat{Y}(\vec{x_k}) \right)^2\right] \approx \frac{1}{\# \text{ of bootstrap}}\sum_{j=1}^{\# \text{ of bootstrap}} (y_k - \hat{Y}_j(\vec{x_k}))^2 = MSE(\vec{x_k})
$$

where the subscript $j$ is used to index the different models. 

We can also compute the ratio of model variance to model risk:

$$
\frac{\text{model variance}}{\text{model risk}}=\frac{\mathrm{Var}(\hat{Y}(\vec{x_k}))}{\mathbb{E}\left[\left(y_k - \hat{Y}(\vec{x_k}) \right)^2\right]}
$$

This ratio provides a sense of how much of model risk can be attributed to model variance, relative to squared bias and irreducible error ($\sigma^2$). You may find it helpful to recall the bias-variance decomposition from lecture:

$$
\text{model risk} = \sigma^2 + (\text{model bias})^2 + \text{model variance}
$$

where $\sigma^2$ is the observational variance, or "irreducible error".

<!-- BEGIN QUESTION -->

<br>

---

### Question 4a

The function `simulate()` below takes in a single data point `xk`, `yk` and a list of models `models`.

- `xk` is a list of feature values for a single data point. `yk` is the scalar outcome.

`simulate()` function should return three quantities: the estimated model risk (`model_risk`), the estimated model variance (`model_var`), and the ratio of model variance to model risk (`ratio`).

We have left one line blank for you to fill in:

```model_risk, model_var, ratio = ...```

Fill in the line correctly to complete the function `simulate`:

In [ ]:
def simulate(xk, yk, models):
    mystery_a = np.array([model.predict([xk])[0] for model in models])
    mystery_b = np.var(mystery_a)
    mystery_c = np.mean((mystery_a - yk) ** 2)
    mystery_d = mystery_b / mystery_c
    model_risk, model_var, ratio = ...
    return model_risk, model_var, ratio

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br>

---

### Question 4b

Using the `simulate` function from above, we can compute the model risk, model variance, and variance-to-risk ratio of **Model B**:

In [ ]:
x_last = X_original[X_original.height - 100]
y_last = Y_original[X_original.height - 100]

(
    partial_feature_model_risk,
    partial_feature_model_var,
    partial_feature_model_ratio
) = simulate(x_last.select(model_b_features).row(0), y_last, partial_feature_models)

print('Model B risk:')
print(partial_feature_model_risk)

print('Model B variance:')
print(partial_feature_model_var)

print('Model B ratio:')
print(partial_feature_model_ratio)

Comment on the variance-to-risk ratio for Model B (`partial_feature_model_ratio`).

- Does the model variance appear to be the dominant term in the bias-variance decomposition? If not, what term(s) dominate the bias-variance decomposition?

Then, given your conclusion above, describe what operation(s) you might perform to reduce the model risk.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>
<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 🔍 Question 5: Logistic Regression

We will explore a breast cancer dataset from the University of Wisconsin ([source](https://archive.ics.uci.edu/ml/datasets/breast+cancer+wisconsin+(diagnostic))). This dataset can be loaded using the `sklearn.datasets.load_breast_cancer()` method. **As a reminder, you should not use LLMs on this assignment except for questions where we specifically suggest using them.**

#### Content Warning
This question includes a discussion about cancer. If you feel uncomfortable with this topic, **please contact your TA, the instructors, or reach out via the [extenuating circumstances form](https://docs.google.com/forms/d/e/1FAIpQLSfu6V-khA3_Gs9KbzupCq_Q9F6QqdR8FMhqeJTcQNqHkzqnfA/viewform).**

In [ ]:
# Run this cell to load the data; no further action is needed.
data = sklearn.datasets.load_breast_cancer()

# Data is a dictionary.
print(data.keys())
print(data.DESCR)

<br/>

Since the data format is a dictionary, we will perform some preprocessing to create a `DataFrame`.

In [ ]:
# Run this cell to see the first five rows of the data; no further action is needed.
df = pl.DataFrame(data.data, schema=list(data.feature_names))
df.head()

The prediction task for this data is to predict whether a tumor is benign or malignant (a binary decision), given the characteristics of that tumor. The prediction labels are stored in the field `data.target`. To put the data back in its original context, we will create a new column called `"malignant"` which will be 1 if the tumor is malignant and 0 if it is benign (reversing the definition of `target`).

In this homework, we will fit a simple **classification model** to predict breast cancer from the cell nuclei of a breast mass. For simplicity, we will work with only one feature: the `mean radius` which corresponds to the size of the tumor. Our output (i.e., response) is the `malignant` column.

In [ ]:
# Run this cell to define X and Y; no further action is needed.

# Target data_dict['target'] = 0 is malignant 1 is benign
df = df.with_columns(pl.Series("malignant", (data.target == 0).astype(int)))

# Define our features/design matrix X
X = df.select("mean radius")
Y = df["malignant"]

<br/>

Before we go further, we will split our dataset into training and testing sets. This lets us explore the prediction power of our trained classifier on both seen and unseen data.

In [ ]:
# Run this cell to create a 75-25 train-test split; no further action is needed. 
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.25, random_state=42)
print(f"Training Data Size: {len(X_train)}")
print(f"Test Data Size: {len(X_test)}")

<br/>
<hr style="border: 1px solid #fdb515;" />

### Part 1: Defining the Model

In these first two parts, you will manually build a logistic regression classifier.

Recall that the Logistic Regression model is written as follows:

$$p = \Large f_{\theta}(x) = \sigma ( x^T \theta)$$

where $f_{\theta}(x)= P(Y=1 | x)$ is the probability that our observation belongs to class 1, and $\sigma$ is the sigmoid activation function:

$$\Large \sigma(z) = \frac{1}{1 + e^{-z}}$$

If we have a single feature, then $x$ is a scalar and our model has parameters $\theta^T = [\theta_0 \ \ \theta_1]$ as follows:

$$\Large f_{\theta}(x) = \sigma (\theta_0 + \theta_1 x)$$

Therefore just like OLS, if we have $n$ data points and $d$ features, then we can construct the design matrix
$$\Large \mathbb{X} \in \mathbb{R}^{n \times (d + 1)}$$
with an all-ones column. Run the below cell to construct `X_intercept_train`. The syntax should look familiar:

In [ ]:
# Run this cell to add the bias column; no further action is needed.
def add_bias_column(X):
    return np.hstack([np.ones((len(X), 1)), X])

X_intercept_train = add_bias_column(X_train)
X_intercept_train.shape

<br/>

---

### Question 5a

Using the above definition for $\mathbb{X}$, we can also construct a matrix representation of our Logistic Regression model, just like we did for OLS. Noting that $\theta^T = [\theta_0 \ \ \theta_1\ \ \dots \ \ \theta_d$], the vector $\hat{\mathbb{Y}}$ is:

$$\Large \hat{\mathbb{Y}} = \sigma(\mathbb{X} \theta) $$

Then the $i$-th element of $\hat{\mathbb{Y}}$ is the probability that the $i$-th observation belongs to class 1, given the feature vector is the $i$-th row of design matrix $\mathbb{X}$, and the parameter vector $\theta$.

Below, implement the `lr_model` function to evaluate this expression. To matrix-multiply two `NumPy` arrays, use `@` or `np.dot`. In case you're interested, the [matmul documentation](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html) contrasts the two methods. **As a reminder, you should not use LLMs on this assignment except for questions where we specifically suggest using them. For question 5, you may only use LLMs on part 5f.**

In [ ]:
def sigmoid(z):
    """
    The sigmoid function is defined for you.
    """
    return 1 / (1 + np.exp(-z))

def lr_model(theta, X):
    """
    Returns the logistic regression model as defined above.
    You should not need to use a for loop; use @ or np.dot.
    
    Args:
        theta: The model parameters. Dimension (d+1,).
        X: The design matrix. Dimension (n, d+1).
    
    Return:
        Probabilities that Y = 1 for each data point.
        Dimension (n,).
    """
    ...

In [ ]:
grader.check("q5a")

<br/>

---

###  Question 5b: Compute Empirical Risk

#### Question 5bi
Now let's try to analyze the cross-entropy loss from logistic regression. Suppose for a single observation, we predict probability $p$ that the true response $y$ is in class 1 (otherwise the prediction is 0 with probability $1 - p$). The cross-entropy loss is $-log(p)$ when $y=1$ and $-log(1-p)$ when $y=0$. More concretely:

$$ \text{CE Loss} = - \left( y \log(p) + (1 - y) \log(1 - p) \right)$$

For the logistic regression model, the **empirical risk** is therefore defined as the average cross-entropy loss across all $n$ data points:

$$R(\theta) = -\frac{1}{n} \sum_{i=1}^n \left( y_i \log(\sigma(X_i^T \theta)) + (1 - y_i) \log(1 - \sigma(X_i^T \theta))  \right) $$

Where $y_i$ is the $i-$th response in our dataset, $\theta$ are the parameters of our model, $X_i^T$ is the $i$-th row of our design matrix $\mathbb{X}$, and $\sigma(X_i^T \theta)$ is the probability that the response is 1 given input $X_i$.

Below, implement the function `lr_avg_loss` that computes empirical risk over the dataset. Feel free to use the functions defined in the previous part.

In [ ]:
def lr_avg_loss(theta, X, Y):
    '''
    Compute the average cross-entropy loss using X, Y, and theta.
    You should not need to use a for loop. 

    Args:
        theta: The model parameters. Dimension (d+1,).
        X: The design matrix. Dimension (n, d+1).
        Y: The label. Dimension (n,).

    Return:
        The average cross-entropy loss.
    '''
    ...
    ...

In [ ]:
grader.check("q5bi")

<br/>

Below is an interactive plot showing the average training cross-entropy loss for various values of $\theta_0$ and $\theta_1$ (respectively x and y axis in the plot).

In [ ]:
# Run this cell to create the plotly visualization; no further action is required. 
from matplotlib import cm

with np.errstate(invalid='ignore', divide='ignore'):
    fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=(10,10))
    
    uvalues = np.linspace(-8,8,70)
    vvalues = np.linspace(-5,5,70)
    u,v = np.meshgrid(uvalues, vvalues)
    thetas = np.vstack((u.flatten(),v.flatten()))
    lr_avg_loss_values = np.array([lr_avg_loss(t, X_intercept_train, Y_train) for t in thetas.T])


    # Plot the surface.
    surf = ax.plot_surface(u, v, np.reshape(lr_avg_loss_values,(len(uvalues), len(vvalues))), 
                           cmap=cm.coolwarm, linewidth=0, antialiased=False)

    # Set the azimuth and elevation angles
    ax.view_init(azim=35, elev=30)
    
    # customize 
    plt.xlabel('theta_0', fontsize=15, labelpad=15)
    plt.ylabel('theta_1', fontsize=15, labelpad=15)
    plt.title('Loss')
    
    # Add a color bar which maps values to colors.
    fig.colorbar(surf, shrink=0.5, aspect=5)
    plt.show()

<!-- BEGIN QUESTION -->

<br/>

---

#### Question 5bii
Describe one interesting observation about the loss plot above.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>
<hr style="border: 1px solid #fdb515;" />

## Part 2: Fit and Predict

### [Tutorial] `scipy.optimize.minimize`

The next two cells call the `minimize` function from `scipy` on the `lr_avg_loss` function you defined in the previous part. We pass in the training data to `args` ([documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)) to find the `theta_hat` that minimizes the average cross-entropy loss over the training set.

In [ ]:
# Run this cell to minimize lr_avg_loss using scipy; no further action is needed.
from scipy.optimize import minimize

min_result = minimize(lr_avg_loss,
                      x0=np.zeros(X_intercept_train.shape[1]),
                      args=(X_intercept_train, Y_train))
min_result

In [ ]:
# Run this cell to print `theta_hat`; no further action is needed.
theta_hat = min_result['x']
theta_hat

<br/>

Because our design matrix $\mathbb{X}$ leads with a column of all ones, `theta_hat` has two elements: $\hat{\theta}_0$ is the estimate of the intercept/bias term, and $\hat{\theta}_1$ is the estimate of the slope of our single feature.

<br/>

### Recap:
* For logistic regression with the parameter vector $\theta$, $P(Y = 1 | x) = \sigma(x^T \theta)$, where $\sigma$ is the sigmoid function and $x$ is a feature vector. Therefore $\sigma(x^T \theta)$ is the probability that the response is 1 given the feature is $x$. Otherwise the response is 0 with probability $P(Y = 0 | x) = 1 - \sigma(x^T \theta)$.
* The $\hat{\theta}$ *minimizes* average cross-entropy loss of our training data.

The main takeaway is that logistic regression models **probabilities** of classifying data points as 1 or 0. Next, we use this takeaway to implement model predictions.

<br/>

---

### Question 5c

Using the `theta_hat` estimate above, we can construct a **decision rule** for classifying a data point with observation $x$. Let $P(Y = 1 | x) = \sigma(x^T \hat{\theta})$:

$$ \text{classify}(x) = \begin{cases} 
1, &\quad\text{if}\ \  P(Y = 1 | x) \geq 0.5 \\ 
0, &\quad\text{if}\ \ P(Y = 1 | x) < 0.5 
\end{cases}$$

This decision rule has a decision **threshold** $T = 0.5$. This threshold means that we treat the classes $0$ and $1$ "equally." Lower thresholds mean that we are more likely to predict $1$, whereas higher thresholds mean that we are more likely to predict $0$.

<br/>

Implement the `lr_predict` function below, which returns a vector of predictions according to the logistic regression model. The function takes a design matrix of observations `X`, parameter estimate `theta`, and decision threshold `threshold` with a default value of 0.5.

In [ ]:
def lr_predict(theta, X, threshold=0.5):
    '''
    Classification using a logistic regression model
    with a given decision rule threshold.

    Args:
        theta: The model parameters. Dimension (d+1,)
        X: The design matrix. Dimension (n, d+1).
        threshold: Decision rule threshold for predicting class 1.

    Return:
        A vector of predictions.
    '''
    ...

# Do not modify below this line.
Y_train_pred = lr_predict(theta_hat, X_intercept_train)
Y_train_pred

In [ ]:
grader.check("q5c")

<br/>

### [Tutorial] Linearly separable data

How do these predicted classifications compare to the true responses $\mathbb{Y}$?

Run the cell below to visualize our predicted responses, the true responses, and the probabilities we used to make predictions. We use `sns.stripplot` which introduces some jitter to avoid overplotting.

In [ ]:
# Run this cell to generate the visualization; no further action is needed.
plot_df = pl.DataFrame({"X": np.squeeze(X_train.to_numpy()),
                        "Y": Y_train,
                        "Y_pred": Y_train_pred,
                        "correct": (Y_train.to_numpy() == Y_train_pred)})
sns.stripplot(data=plot_df.to_pandas(), x="X", y="Y", orient='h', alpha=0.8, hue="correct")
plt.xlabel('mean radius, $x$')
plt.ylabel('$y$')
plt.yticks(ticks=[0, 1], labels=['0:\nbenign', '1:\nmalignant'])
plt.title("Predictions for decision threshold T = 0.5")
plt.show()

<br/>

Because we are using a decision threshold $T = 0.5$, we predict $1$ for all $x$ where our predicted probability $\sigma(x^T\theta)$ is greater than or equal to 0.5. Writing this out mathematically and solving for $x^T\theta$: 
\begin{align}
\frac{1}{1+e^{-x^T\theta}} &= \frac{1}{2} \\ 
2 &= 1 + e^{-x^T\theta} \\
1 &= e^{-x^T\theta} \\
log(1)  &= log(e^{-x^T\theta})  \\
0 &= -x^T\theta  \\
\end{align}

We see that our decision threshold is when $x^T\theta = 0$. For the single mean radius feature, we can use algebra to solve for the boundary to be approximately $x \approx 14.8$. We can see this by substituting for $\theta = \hat{\theta}$ in the equation above:
$$x^T\hat{\theta} = 0 $$
$$
\begin{matrix}\begin{bmatrix}1 & x\end{bmatrix}\\\mbox{}\end{matrix}\begin{bmatrix} \hat{\theta_0} \\ \hat{\theta_1} \end{bmatrix} = 0\\
$$

From the minimize function, we found that `theta_hat` is `array([-13.87178543,   0.93723909])`. Plugging for $\hat{\theta}$:

$$-13.87178543 +  0.93723909x = 0$$ 

$$x \approx 14.8$$


In other words, we will always predict $0$ (benign) if the mean radius feature is less than 14.8 and $1$ (malignant) otherwise. However, in our training data, there are data points with large mean radii that are benign and vice versa. Our data is not **linearly separable** by a vertical line.

The above visualization is useful when we have just one feature. In practice, however, we use other performance metrics to diagnose our model performance. Next, we will explore several metrics: accuracy, precision, recall, and confusion matrices.

<br/>
<hr style="border: 1px solid #fdb515;" />

### Part 3: Quantifying Performance


### [Tutorial] sklearn's `LogisticRegression`
For this question, we will use `sklearn`'s `LogisticRegression` function, which operates similarly to the `sklearn` OLS, Ridge, and LASSO models.

Let's first fit a logistic regression model to the training data. Some notes: 
* Like with linear models, the `fit_intercept` argument specifies if the model includes an intercept term. We therefore pass in the original matrix `X_train` (defined at the beginning of the notebook, without intercept term) in the call to `lr.fit()`.
* `sklearn` fits an **L2 regularized** logistic regression model by default as specified in the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) for more details. The `penalty` argument specifies the regularization penalty term.

In [ ]:
# Run this cell to fit a sklearn LogisticRegression model; no further action is needed. 
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
        fit_intercept=True,
        penalty='l2')

lr.fit(X_train, Y_train)
lr.intercept_, lr.coef_


Like with linear models, we can call `lr.predict(X_train)` to classify our training data with our fitted model.

In [ ]:
# Run this cell to make predictions; no further action is needed. 
lr.predict(X_train)

Note that for a binary classification task, the `sklearn` model uses an unadjustable decision rule of 0.5. If you're interested in manually adjusting this threshold, check out the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict_proba) for `lr.predict_proba()`.

<br/>

---

### Question 5d: Accuracy

Fill in the code below to compute the training and testing accuracy, defined as:

$$
\text{Training Accuracy} = \frac{1}{n_{train\_set}} \sum_{i \in {train\_set}} {\mathbb{1}_{y_i = \hat{y_i}}}
$$

$$
\text{Testing Accuracy} = \frac{1}{n_{test\_set}} \sum_{i \in {test\_set}} {\mathbb{1}_{y_i = \hat{y_i}}}
$$

where for the $i$-th observation in the respective dataset, $\hat{y_i}$ is the predicted response (class 0 or 1), and $y_i$ is the true response.  $\mathbb{1}_{y_i = \hat{y_i}}$ is an indicator function which is $1$ if ${y_i} = \hat{y_i}$ and $ 0$ otherwise.

**Hint**: `lr.predict` hands back a `NumPy` array, so call `to_numpy()` ([documentation](https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.to_numpy.html)) on the true responses before comparing the two.

In [ ]:
train_accuracy = ...
test_accuracy = ...

print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

In [ ]:
grader.check("q5d")

<br/>

---

### Question 5e: Precision and Recall

It seems we can get a very high test accuracy. What about precision and recall?  
- **Precision** (also called positive predictive value) is the fraction of true positives among the total number of data points predicted as positive.  
- **Recall** (also known as sensitivity) is the fraction of true positives among the total number of data points with positive labels.

Precision measures the ability of our classifier to avoid predicting negative samples as positive (i.e., avoid false positives), while recall is the ability of the classifier to find all the positive samples (i.e., avoid false negatives).

Mathematically, Precision and Recall are defined as:
$$
\text{Precision} = \frac{n_{true\_positives}}{n_{true\_positives} + n_{false\_positives}} = \frac{TP}{TP+FP}
$$

$$
\text{Recall} = \frac{n_{true\_positives}}{n_{true\_positives} + n_{false\_negatives}}=\frac{TP}{TP+FN}
$$

Use the formulas above to compute the precision and recall for the **test set** using the `lr` model trained using `sklearn`.

In [ ]:
Y_test_pred = ...

precision = ...
recall = ...

print(f'precision = {precision:.4f}')
print(f'recall = {recall:.4f}')

In [ ]:
grader.check("q5e")

<br/>

Our precision is fairly high, while our recall is a bit lower.

Consider the following plots, which display the distribution of the **response variable** $\mathbb{Y}$ in the training and test sets. Recall class labels are 0: benign, 1: malignant.

In [ ]:
fig, axes = plt.subplots(1, 2)
sns.countplot(x=Y_train, ax=axes[0]);
sns.countplot(x=Y_test, ax=axes[1]);

axes[0].set_title('Train')
axes[1].set_title('Test')
plt.tight_layout();

<!-- BEGIN QUESTION -->

<br>

---

### Question 5f
Now, with the help of an LLM, generate the following: 1) a precision-recall curve that plots recall and precision at varying thresholds, and 2) an ROC curve that plots the false positive rate and true positive rate at varying thresholds. Use `plt.subplots` to create two plots side by side.

In [ ]:
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br>

---

### Question 5g
Refer to both curves you created in the previous subpart. In 4 to 6 sentences:

What happens to precision and recall as the threshold increases, and what happens to the false positive rate and true positive rate? Why? 

Using your explanation, explain the tradeoffs a doctor might face when choosing a threshold above which to classify tumors as malignant. Would you choose a threshold below or above 0.5? Justify your choice.



_Type your answer here, replacing this text._

<!-- END QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Congratulations! You have finished Homework 7!

## Course Content Feedback

If you have any feedback about this assignment or about any of our other assignments, lectures, or discussions, please fill out the [Course Content Feedback Form](https://docs.google.com/forms/d/e/1FAIpQLSfN9C-RJoe9hD8G2sbd1reJQh5H4WwKLFFrEt4DeQUBKmToJQ/viewform). Your input is valuable in helping us improve the quality and relevance of our content to better meet your needs and expectations!

## Submission Instructions

Below, you will see a cell. Running this cell will automatically generate a zip file with your autograded answers. Once you submit this file to the HW 7 Coding assignment on Pensive, Pensive will automatically submit a PDF file with your written answers to the HW 7 Coding Written assignment. If you run into any issues when running this cell, feel free to check this [section](https://ds100.org/debugging-guide/autograder-pensive/) in the Data 100 Debugging Guide.
                                                                                                                                                        
**Important**: Please check that your **plots/graphs and written responses** were generated and submitted correctly to the HW 7 Coding Written Assignment. **Please also make sure you have submitted your responses to the written work to the HW 7 Handwritten Work Assignment.**  

**You are responsible for ensuring your submission follows our requirements and that the PDF for HW 7 written answers was generated/submitted correctly. We will not be granting regrade requests nor extensions to submissions that don't follow instructions.** If you encounter any difficulties with submission, please don't hesitate to reach out to staff prior to the deadline.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)